# Train LightGBM Ranker - Restaurant Recommendation
Notebook này tách theo từng chức năng rõ ràng: preprocessing, feature engineering, validation, training, save model.

Các điểm đã chuẩn hóa theo data mới:
- Cột ngày chỉ giữ giờ mở và giờ đóng (bỏ True/False).
- Dùng `Ngày nghỉ` để suy ra `Ngày hoạt động`.
- `LstTargetAudience`, `LstCategory` tách bằng regex `||` thành array thật.
- Bổ sung feature so sánh ngày và giờ mở cửa theo query cho model.

In [7]:
# 1) Imports + config
import math
import re
import unicodedata
from collections import Counter
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import ndcg_score
from sklearn.model_selection import train_test_split

RAW_FOODY_PATH = Path("dataset/foody_combined_data_final.csv")
LABELED_PATH = Path("dataset/restaurant_dataset_ver1.csv")
MODEL_OUT = Path("lgbm_ranker_best.txt")

RANDOM_STATE = 42
VALID_SIZE = 0.2

LABEL_COL = "relevance_score"
QUERY_COL = "query_id"

FULL_DAYS = ["Chủ nhật", "Thứ hai", "Thứ ba", "Thứ tư", "Thứ năm", "Thứ sáu", "Thứ bảy"]

CATEGORICAL_COLS = [
    "District",
    "Phan_loai_mon",
    "Has_day_constraint",
    "Has_time_constraint",
    "Open_time_match",
]

NUMERICAL_COLS = [
    "Price",
    "Pinecone_score",
    "BM25_score",
    "Semantic_sim_score",
    "District_match_score",
    "Category_match_score",
    "Price_budget_match",
    "Price_distance_norm",
    "Popularity_score",
    "Service_match_score",
    "Open_day_match_ratio",
    "Open_time_distance_min",
]

FEATURE_COLS = CATEGORICAL_COLS + NUMERICAL_COLS

In [8]:
# 2) Utility functions + preprocessing helpers
def normalize_restaurant_id(series: pd.Series) -> pd.Series:
    s = series.astype(str).str.strip()
    s = s.replace({"nan": "", "None": ""})
    num = pd.to_numeric(s, errors="coerce")
    out = s.copy()
    mask = num.notna()
    out.loc[mask] = num.loc[mask].astype(int).astype(str)
    return out


def normalize_text(text: object) -> str:
    s = str(text or "").lower().strip()
    s = unicodedata.normalize("NFD", s)
    s = "".join(ch for ch in s if unicodedata.category(ch) != "Mn")
    s = re.sub(r"[^a-z0-9\s-]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def tokenize(text: str) -> list[str]:
    return re.findall(r"[a-z0-9]+", text)


def split_pipe_values(value: object) -> list[str]:
    if pd.isna(value):
        return []
    text = str(value).strip()
    if not text:
        return []
    return [x.strip() for x in re.split(r"\s*\|\|\s*", text) if x.strip()]


def get_active_days(off_value: object) -> list[str]:
    if pd.isna(off_value) or str(off_value).strip() == "":
        return FULL_DAYS.copy()
    off_days = set(split_pipe_values(off_value))
    return [d for d in FULL_DAYS if d not in off_days]


def normalize_open_close(raw: object) -> tuple[str, str] | None:
    text = "" if pd.isna(raw) else str(raw).strip()
    if not text:
        return None

    # Handles both formats: ('08:00', '22:00') and (('08:00', '22:00'), False)
    m = re.search(r"'([0-2]\d:[0-5]\d)'\s*,\s*'([0-2]\d:[0-5]\d)'", text)
    if not m:
        return None
    return m.group(1), m.group(2)


def to_minutes(hhmm: str) -> int:
    h, m = hhmm.split(":")
    return int(h) * 60 + int(m)


def parse_budget_vnd(query: str) -> float:
    q = normalize_text(query)
    m_k = re.search(r"(\d+[\.,]?\d*)\s*(k|nghin)", q)
    if m_k:
        return float(m_k.group(1).replace(",", ".")) * 1000.0

    m_trieu = re.search(r"(\d+[\.,]?\d*)\s*(trieu|tr)", q)
    if m_trieu:
        return float(m_trieu.group(1).replace(",", ".")) * 1_000_000.0

    m_raw = re.search(r"(\d{5,8})", q)
    if m_raw:
        return float(m_raw.group(1))

    return -1.0


DAY_PATTERNS = {
    "Chủ nhật": [r"\bch[uủ]\s*nh[aậ]t\b", r"\bcn\b"],
    "Thứ hai": [r"\bth[uứ]\s*2\b", r"\bth[uứ]\s*hai\b", r"\bt2\b"],
    "Thứ ba": [r"\bth[uứ]\s*3\b", r"\bth[uứ]\s*ba\b", r"\bt3\b"],
    "Thứ tư": [r"\bth[uứ]\s*4\b", r"\bth[uứ]\s*t[uư]\b", r"\bt4\b"],
    "Thứ năm": [r"\bth[uứ]\s*5\b", r"\bth[uứ]\s*n[aă]m\b", r"\bt5\b"],
    "Thứ sáu": [r"\bth[uứ]\s*6\b", r"\bth[uứ]\s*s[aá]u\b", r"\bt6\b"],
    "Thứ bảy": [r"\bth[uứ]\s*7\b", r"\bth[uứ]\s*b[aả]y\b", r"\bt7\b"],
}

HOUR_PATTERN = re.compile(r"\b([01]?\d|2[0-3])(?:[:h]([0-5]?\d))?\b")


def extract_query_days(query_text: str) -> set[str]:
    q = query_text.lower()
    found = set()
    for day, pats in DAY_PATTERNS.items():
        if any(re.search(p, q) for p in pats):
            found.add(day)
    return found


def extract_query_minutes(query_text: str) -> list[int]:
    q = query_text.lower()
    minutes = []
    for h_txt, m_txt in HOUR_PATTERN.findall(q):
        h = int(h_txt)
        m = int(m_txt) if m_txt else 0
        minutes.append(h * 60 + m)

    if "sáng" in q:
        minutes.append(8 * 60)
    if "trưa" in q:
        minutes.append(12 * 60)
    if "chiều" in q:
        minutes.append(15 * 60)
    if "tối" in q:
        minutes.append(20 * 60)
    if "đêm" in q:
        minutes.append(22 * 60)

    return sorted(set(minutes))

In [9]:
# 3) Feature engineering + build training dataframe
class SimpleBM25:
    def __init__(self, documents: list[list[str]], k1: float = 1.5, b: float = 0.75):
        self.k1 = k1
        self.b = b
        self.documents = [list(doc) for doc in documents]
        self.doc_count = len(self.documents)
        self.doc_lens = [len(doc) for doc in self.documents]
        self.avgdl = float(np.mean(self.doc_lens)) if self.doc_lens else 0.0

        self.term_freqs: list[Counter] = []
        self.doc_freqs: Counter = Counter()

        for doc in self.documents:
            tf = Counter(doc)
            self.term_freqs.append(tf)
            for term in tf.keys():
                self.doc_freqs[term] += 1

    def idf(self, term: str) -> float:
        df = self.doc_freqs.get(term, 0)
        return math.log(1.0 + (self.doc_count - df + 0.5) / (df + 0.5))

    def score(self, query_tokens: list[str], doc_idx: int) -> float:
        if self.doc_count == 0:
            return 0.0

        tf = self.term_freqs[doc_idx]
        dl = self.doc_lens[doc_idx]
        score = 0.0
        for term in query_tokens:
            freq = tf.get(term, 0)
            if freq == 0:
                continue
            idf = self.idf(term)
            denom = freq + self.k1 * (1 - self.b + self.b * (dl / (self.avgdl + 1e-9)))
            score += idf * ((freq * (self.k1 + 1.0)) / (denom + 1e-9))
        return float(score)


def compute_semantic_similarity(query_series: pd.Series, name_series: pd.Series, category_series: pd.Series) -> np.ndarray:
    query_texts = query_series.fillna("").astype(str).tolist()
    doc_texts = (name_series.fillna("").astype(str) + " | " + category_series.fillna("").astype(str)).tolist()

    try:
        from dataset.embed_model import embed_text

        unique_queries = list(dict.fromkeys(query_texts))
        unique_docs = list(dict.fromkeys(doc_texts))

        def batched_embed(texts: list[str], batch_size: int = 64) -> np.ndarray:
            if not texts:
                return np.zeros((0, 1), dtype=np.float32)
            chunks = []
            for i in range(0, len(texts), batch_size):
                emb = embed_text(texts[i : i + batch_size]).cpu().numpy().astype(np.float32)
                chunks.append(emb)
            return np.vstack(chunks)

        q_emb = batched_embed(unique_queries)
        d_emb = batched_embed(unique_docs)

        q_emb = q_emb / (np.linalg.norm(q_emb, axis=1, keepdims=True) + 1e-12)
        d_emb = d_emb / (np.linalg.norm(d_emb, axis=1, keepdims=True) + 1e-12)

        q_map = {q: i for i, q in enumerate(unique_queries)}
        d_map = {d: i for i, d in enumerate(unique_docs)}

        sims = np.zeros(len(query_texts), dtype=np.float32)
        for i, (q, d) in enumerate(zip(query_texts, doc_texts)):
            sims[i] = float(np.dot(q_emb[q_map[q]], d_emb[d_map[d]]))
        return sims

    except Exception as exc:
        print(f"[WARN] Semantic embedding unavailable, fallback to lexical similarity: {exc}")
        sims = []
        for q, d in zip(query_texts, doc_texts):
            q_tok = set(tokenize(normalize_text(q)))
            d_tok = set(tokenize(normalize_text(d)))
            if not q_tok and not d_tok:
                sims.append(0.0)
            else:
                sims.append(len(q_tok & d_tok) / max(len(q_tok | d_tok), 1))
        return np.array(sims, dtype=np.float32)


def category_match_score(query_text: str, categories: list[str]) -> float:
    q_tokens = set(tokenize(normalize_text(query_text)))
    c_tokens = set(tokenize(normalize_text(" ".join(categories))))
    if not c_tokens:
        return 0.0
    return len(q_tokens & c_tokens) / float(len(c_tokens))


def district_match_score(query_text: str, district_text: object) -> float:
    q_norm = normalize_text(query_text)
    d_norm = normalize_text(district_text)
    if not d_norm:
        return 0.0
    return 1.0 if d_norm in q_norm else 0.0


def parse_bool_flag(value: object) -> bool:
    return str(value).strip().lower() in {"1", "1.0", "true", "yes"}


def service_match_score(query_text: str, delivery_flag: object, booking_flag: object) -> float:
    q = normalize_text(query_text)
    need_delivery = ("giao" in q and "hang" in q) or ("ship" in q)
    need_booking = ("dat ban" in q)

    delivery_ok = parse_bool_flag(delivery_flag)
    booking_ok = parse_bool_flag(booking_flag)

    score = 0.0
    if need_delivery:
        score += 1.0 if delivery_ok else -0.5
    if need_booking:
        score += 1.0 if booking_ok else -0.5
    return score


def day_time_match_features(query_text: str, open_map: dict[str, tuple[str, str] | None], active_days: list[str]) -> dict[str, float]:
    req_days = extract_query_days(query_text)
    req_minutes = extract_query_minutes(query_text)

    has_day_constraint = 1.0 if req_days else 0.0
    has_time_constraint = 1.0 if req_minutes else 0.0

    selected_days = req_days if req_days else set(FULL_DAYS)
    selected_days = [d for d in selected_days if d in active_days]

    if selected_days:
        open_day_ratio = len(selected_days) / max(len(req_days), 1) if req_days else len(active_days) / 7.0
    else:
        open_day_ratio = 0.0

    if not req_minutes:
        return {
            "Has_day_constraint": has_day_constraint,
            "Has_time_constraint": has_time_constraint,
            "Open_day_match_ratio": float(open_day_ratio),
            "Open_time_match": 0.0,
            "Open_time_distance_min": float(24 * 60),
        }

    best_distance = 24 * 60
    open_time_match = 0.0

    for day in selected_days:
        oc = open_map.get(day)
        if oc is None:
            continue

        open_m = to_minutes(oc[0])
        close_m = to_minutes(oc[1])
        if close_m <= open_m:
            close_m += 24 * 60

        for t in req_minutes:
            tt = t
            if tt < open_m:
                tt += 24 * 60

            if open_m <= tt <= close_m:
                open_time_match = 1.0
                best_distance = 0
            else:
                best_distance = min(best_distance, abs(tt - open_m), abs(tt - close_m))

    return {
        "Has_day_constraint": has_day_constraint,
        "Has_time_constraint": has_time_constraint,
        "Open_day_match_ratio": float(open_day_ratio),
        "Open_time_match": float(open_time_match),
        "Open_time_distance_min": float(best_distance),
    }


def preprocess_raw_dataframe(df_raw: pd.DataFrame) -> pd.DataFrame:
    df_raw = df_raw.copy()

    if "LstTargetAudience" in df_raw.columns:
        df_raw["LstTargetAudience_arr"] = df_raw["LstTargetAudience"].apply(split_pipe_values)
    else:
        df_raw["LstTargetAudience_arr"] = [[] for _ in range(len(df_raw))]

    if "LstCategory" in df_raw.columns:
        df_raw["LstCategory_arr"] = df_raw["LstCategory"].apply(split_pipe_values)
    else:
        df_raw["LstCategory_arr"] = [[] for _ in range(len(df_raw))]

    if "Ngày nghỉ" in df_raw.columns:
        df_raw["Ngày hoạt động"] = df_raw["Ngày nghỉ"].apply(get_active_days)
    else:
        df_raw["Ngày hoạt động"] = [FULL_DAYS.copy() for _ in range(len(df_raw))]

    for day in FULL_DAYS:
        if day in df_raw.columns:
            df_raw[f"{day}_open_close"] = df_raw[day].apply(normalize_open_close)
            df_raw[day] = df_raw[f"{day}_open_close"].apply(
                lambda x: f"{x[0]}-{x[1]}" if isinstance(x, tuple) else ""
            )

    return df_raw


def build_training_dataframe(raw_foody_path: Path, labeled_path: Path) -> pd.DataFrame:
    df_raw = pd.read_csv(raw_foody_path)
    df_lbl = pd.read_csv(labeled_path)

    required_lbl = {"query", "restaurant_id", "label", "source"}
    missing_lbl = sorted(required_lbl - set(df_lbl.columns))
    if missing_lbl:
        raise ValueError(f"Missing columns in labeled file: {missing_lbl}")

    required_raw = {"RestaurantID", "Name", "District", "PriceMin", "PriceMax", "Giao tận nơi", "Đặt bàn"}
    missing_raw = sorted(required_raw - set(df_raw.columns))
    if missing_raw:
        raise ValueError(f"Missing columns in raw file: {missing_raw}")

    df_raw = preprocess_raw_dataframe(df_raw)
    df_lbl = df_lbl.copy()

    df_lbl["restaurant_id_norm"] = normalize_restaurant_id(df_lbl["restaurant_id"])
    df_raw["restaurant_id_norm"] = normalize_restaurant_id(df_raw["RestaurantID"])

    keep_cols = [
        "restaurant_id_norm",
        "Name",
        "District",
        "LstCategory_arr",
        "PriceMin",
        "PriceMax",
        "TotalView",
        "TotalFavourite",
        "TotalCheckins",
        "Giao tận nơi",
        "Đặt bàn",
        "Ngày hoạt động",
        *[f"{d}_open_close" for d in FULL_DAYS],
    ]
    keep_cols = [c for c in keep_cols if c in df_raw.columns]

    df_merge = df_lbl.merge(df_raw[keep_cols], on="restaurant_id_norm", how="left")
    if df_merge["District"].isna().all():
        raise ValueError("Merge failed: no matched restaurants from raw dataset")

    q_norm = df_merge["query"].apply(normalize_text)
    name_norm = df_merge["Name"].fillna("").astype(str).apply(normalize_text)

    bm25_docs = name_norm.apply(tokenize).tolist()
    bm25 = SimpleBM25(bm25_docs)
    bm25_scores = [bm25.score(tokenize(q), i) for i, q in enumerate(q_norm.tolist())]

    category_text_series = df_merge["LstCategory_arr"].apply(
        lambda x: " ".join(x) if isinstance(x, list) else ""
    )
    semantic_sim_scores = compute_semantic_similarity(
        query_series=df_merge["query"],
        name_series=df_merge["Name"].fillna(""),
        category_series=category_text_series,
    )

    price_min = pd.to_numeric(df_merge.get("PriceMin", 0), errors="coerce").fillna(0.0)
    price_max = pd.to_numeric(df_merge.get("PriceMax", 0), errors="coerce").fillna(0.0)
    price_mid = (price_min + price_max) / 2.0

    budgets = df_merge["query"].apply(parse_budget_vnd)
    has_budget = budgets > 0
    price_budget_match = np.where(has_budget & (price_mid <= budgets), 1.0, 0.0)
    price_distance_norm = np.where(
        has_budget,
        np.maximum(price_mid - budgets, 0.0) / np.maximum(budgets, 1.0),
        0.0,
    )

    total_view = pd.to_numeric(df_merge.get("TotalView", 0), errors="coerce").fillna(0.0)
    total_fav = pd.to_numeric(df_merge.get("TotalFavourite", 0), errors="coerce").fillna(0.0)
    total_checkin = pd.to_numeric(df_merge.get("TotalCheckins", 0), errors="coerce").fillna(0.0)
    popularity_score = np.log1p(total_view) + 0.5 * np.log1p(total_fav) + 0.5 * np.log1p(total_checkin)

    day_time_features = []
    for _, row in df_merge.iterrows():
        open_map = {day: row.get(f"{day}_open_close", None) for day in FULL_DAYS}
        active_days = row.get("Ngày hoạt động", FULL_DAYS)
        if not isinstance(active_days, list):
            active_days = FULL_DAYS
        day_time_features.append(day_time_match_features(str(row["query"]), open_map, active_days))

    day_time_df = pd.DataFrame(day_time_features)

    df = pd.DataFrame({
        QUERY_COL: pd.factorize(df_merge["query"], sort=False)[0].astype(np.int32),
        LABEL_COL: pd.to_numeric(df_merge["label"], errors="coerce"),
        "District": df_merge["District"].fillna("Khong_xac_dinh").astype(str),
        "Phan_loai_mon": df_merge["LstCategory_arr"].apply(
            lambda x: " | ".join(x) if isinstance(x, list) and len(x) > 0 else "Khong_xac_dinh"
        ),
        "Price": price_mid,
        "Pinecone_score": (df_merge["source"].astype(str).str.lower() == "pinecone").astype(float),
        "BM25_score": bm25_scores,
        "Semantic_sim_score": semantic_sim_scores,
        "District_match_score": df_merge.apply(
            lambda r: district_match_score(r["query"], r["District"]), axis=1
        ),
        "Category_match_score": df_merge.apply(
            lambda r: category_match_score(r["query"], r["LstCategory_arr"] if isinstance(r["LstCategory_arr"], list) else []), axis=1
        ),
        "Price_budget_match": price_budget_match.astype(float),
        "Price_distance_norm": price_distance_norm.astype(float),
        "Popularity_score": popularity_score.astype(float),
        "Service_match_score": df_merge.apply(
            lambda r: service_match_score(r["query"], r.get("Giao tận nơi", ""), r.get("Đặt bàn", "")), axis=1
        ),
    })

    df = pd.concat([df, day_time_df], axis=1)

    for col in NUMERICAL_COLS + [LABEL_COL]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    for col in CATEGORICAL_COLS:
        df[col] = df[col].fillna("unknown").astype(str).astype("category")

    for col in NUMERICAL_COLS:
        med = df[col].median(skipna=True)
        fill_value = float(med) if not np.isnan(med) else 0.0
        df[col] = df[col].fillna(fill_value)

    df = df.dropna(subset=[LABEL_COL, QUERY_COL]).copy()

    labels = df[LABEL_COL].to_numpy()
    if np.any(labels < 0) or np.any(labels > 4):
        raise ValueError("label must be in [0, 4]")

    df[LABEL_COL] = df[LABEL_COL].astype(int)
    return df


def validate_preprocessing(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    total_rows = len(df)
    for col in FEATURE_COLS + [LABEL_COL, QUERY_COL]:
        miss = df[col].isna().sum() if col in df.columns else total_rows
        dtype = str(df[col].dtype) if col in df.columns else "missing"
        rows.append({
            "feature": col,
            "dtype": dtype,
            "missing": int(miss),
            "missing_ratio": float(miss / max(total_rows, 1)),
        })

    report = pd.DataFrame(rows).sort_values(["missing_ratio", "feature"], ascending=[False, True]).reset_index(drop=True)
    print("[INFO] Preprocessing validation summary")
    print(f"Rows: {total_rows:,}")
    print(f"Unique queries: {df[QUERY_COL].nunique():,}")
    print(f"Label distribution:\n{df[LABEL_COL].value_counts(dropna=False).sort_index()}")
    return report


def build_group_array(data: pd.DataFrame) -> np.ndarray:
    return data.groupby(QUERY_COL, sort=False).size().to_numpy(dtype=np.int32)


def split_by_query(df: pd.DataFrame, valid_size: float = 0.2, random_state: int = 42):
    unique_queries = df[QUERY_COL].drop_duplicates().to_numpy()
    train_q, valid_q = train_test_split(
        unique_queries,
        test_size=valid_size,
        random_state=random_state,
        shuffle=True,
    )

    train_df = df[df[QUERY_COL].isin(train_q)].copy()
    valid_df = df[df[QUERY_COL].isin(valid_q)].copy()

    if train_df.empty or valid_df.empty:
        raise ValueError("Train/valid split is empty. Check valid_size or number of queries")

    return train_df, valid_df


def querywise_ndcg_at_k(df_valid: pd.DataFrame, preds: np.ndarray, k: int = 5) -> float:
    temp = df_valid[[QUERY_COL, LABEL_COL]].copy()
    temp["pred"] = preds

    scores = []
    for _, g in temp.groupby(QUERY_COL, sort=False):
        y_true = g[LABEL_COL].to_numpy().reshape(1, -1)
        y_pred = g["pred"].to_numpy().reshape(1, -1)
        scores.append(ndcg_score(y_true, y_pred, k=k))

    return float(np.mean(scores))

In [10]:
# 4) Build dataset + validate preprocessing + split
df = build_training_dataframe(RAW_FOODY_PATH, LABELED_PATH)
validation_report = validate_preprocessing(df)
display(validation_report.head(20))

train_df, valid_df = split_by_query(df, valid_size=VALID_SIZE, random_state=RANDOM_STATE)

x_train = train_df[FEATURE_COLS]
y_train = train_df[LABEL_COL].astype(int)
g_train = build_group_array(train_df)

x_valid = valid_df[FEATURE_COLS]
y_valid = valid_df[LABEL_COL].astype(int)
g_valid = build_group_array(valid_df)

print(f"Train rows: {len(train_df):,} | Train queries: {len(g_train):,}")
print(f"Valid rows: {len(valid_df):,} | Valid queries: {len(g_valid):,}")
print(f"Total unique queries: {df[QUERY_COL].nunique():,}")

[INFO] Preprocessing validation summary
Rows: 7,500
Unique queries: 500
Label distribution:
relevance_score
0    3908
1    1602
2     675
3     594
4     721
Name: count, dtype: int64


,feature,dtype,missing,missing_ratio
0,BM25_score,float64,0,0.0
1,Category_match_score,float64,0,0.0
2,District,category,0,0.0
3,District_match_score,float64,0,0.0
4,Has_day_constraint,category,0,0.0
5,Has_time_constraint,category,0,0.0
6,Open_day_match_ratio,float64,0,0.0
7,Open_time_distance_min,float64,0,0.0
8,Open_time_match,category,0,0.0
9,Phan_loai_mon,category,0,0.0


Train rows: 6,000 | Train queries: 400
Valid rows: 1,500 | Valid queries: 100
Total unique queries: 500


In [11]:
# 5) Train with small grid search
param_grid = [
    {"max_depth": 4, "num_leaves": 15},
    {"max_depth": 4, "num_leaves": 20},
    {"max_depth": 5, "num_leaves": 20},
    {"max_depth": 6, "num_leaves": 31},
]

results = []
best_model = None
best_score = -1.0

for p in param_grid:
    print("\n" + "=" * 70)
    print(f"Training with max_depth={p['max_depth']}, num_leaves={p['num_leaves']}")

    ranker = lgb.LGBMRanker(
        objective="rank_xendcg",
        n_estimators=3000,
        learning_rate=0.05,
        max_depth=p["max_depth"],
        num_leaves=p["num_leaves"],
        min_child_samples=20,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=RANDOM_STATE,
    )

    ranker.fit(
        x_train,
        y_train,
        group=g_train,
        eval_set=[(x_valid, y_valid)],
        eval_group=[g_valid],
        eval_metric="ndcg",
        eval_at=[5],
        categorical_feature=CATEGORICAL_COLS,
        callbacks=[
            lgb.early_stopping(stopping_rounds=100, first_metric_only=True, verbose=False),
            lgb.log_evaluation(period=100),
        ],
    )

    best_iter = ranker.best_iteration_
    preds_valid = ranker.predict(x_valid, num_iteration=best_iter)
    manual_ndcg5 = querywise_ndcg_at_k(valid_df, preds_valid, k=5)
    lgb_ndcg5 = ranker.best_score_.get("valid_0", {}).get("ndcg@5")

    row = {
        "max_depth": p["max_depth"],
        "num_leaves": p["num_leaves"],
        "best_iteration": int(best_iter) if best_iter is not None else None,
        "lightgbm_valid_ndcg@5": float(lgb_ndcg5) if lgb_ndcg5 is not None else np.nan,
        "manual_querywise_ndcg@5": float(manual_ndcg5),
    }
    results.append(row)

    if manual_ndcg5 > best_score:
        best_score = manual_ndcg5
        best_model = ranker

results_df = pd.DataFrame(results).sort_values(
    "manual_querywise_ndcg@5", ascending=False
).reset_index(drop=True)
display(results_df)


Training with max_depth=4, num_leaves=15
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000851 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1201
[LightGBM] [Info] Number of data points in the train set: 6000, number of used features: 16
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[Light

,max_depth,num_leaves,best_iteration,lightgbm_valid_ndcg@5,manual_querywise_ndcg@5
0,6,31,238,0.746017,0.747674
1,4,20,272,0.747897,0.744340
2,5,20,155,0.744154,0.740418
3,4,15,272,0.743889,0.739694


In [12]:
# 6) Save best model
if best_model is None:
    raise RuntimeError("No model was trained")

MODEL_OUT.parent.mkdir(parents=True, exist_ok=True)
best_model.booster_.save_model(str(MODEL_OUT))

print("Best config:")
print(results_df.iloc[0])
print(f"\nSaved best model to: {MODEL_OUT.resolve()}")

Best config:
max_depth                    6.000000
num_leaves                  31.000000
best_iteration             238.000000
lightgbm_valid_ndcg@5        0.746017
manual_querywise_ndcg@5      0.747674
Name: 0, dtype: float64

Saved best model to: C:\Users\admin\Desktop\Restaurant-recommendation-system\lgbm_ranker_best.txt


In [13]:
# 7) Inference utilities: rank top restaurants by query
def load_ranker_for_inference():
    if "best_model" in globals() and best_model is not None:
        return best_model.booster_
    if MODEL_OUT.exists():
        return lgb.Booster(model_file=str(MODEL_OUT))
    raise RuntimeError("No trained model found. Run training and save model first.")


def build_inference_features(query_text: str, raw_foody_path: Path = RAW_FOODY_PATH) -> pd.DataFrame:
    df_raw = pd.read_csv(raw_foody_path)
    df_raw = preprocess_raw_dataframe(df_raw)

    required_raw = {"RestaurantID", "Name", "District", "PriceMin", "PriceMax", "Giao tận nơi", "Đặt bàn"}
    missing_raw = sorted(required_raw - set(df_raw.columns))
    if missing_raw:
        raise ValueError(f"Missing columns in raw file: {missing_raw}")

    df_inf = df_raw.copy()
    df_inf["restaurant_id_norm"] = normalize_restaurant_id(df_inf["RestaurantID"])
    df_inf["restaurant_name"] = df_inf["Name"].fillna("").astype(str)
    df_inf["District"] = df_inf["District"].fillna("Khong_xac_dinh").astype(str)
    df_inf["Phan_loai_mon"] = df_inf["LstCategory_arr"].apply(
        lambda x: " | ".join(x) if isinstance(x, list) and len(x) > 0 else "Khong_xac_dinh"
    )

    q_series = pd.Series([query_text] * len(df_inf))
    name_norm = df_inf["restaurant_name"].apply(normalize_text)
    bm25_docs = name_norm.apply(tokenize).tolist()
    bm25 = SimpleBM25(bm25_docs)
    q_norm = normalize_text(query_text)
    bm25_scores = [bm25.score(tokenize(q_norm), i) for i in range(len(df_inf))]

    category_text_series = df_inf["LstCategory_arr"].apply(
        lambda x: " ".join(x) if isinstance(x, list) else ""
    )
    semantic_sim_scores = compute_semantic_similarity(
        query_series=q_series,
        name_series=df_inf["restaurant_name"],
        category_series=category_text_series,
    )

    price_min = pd.to_numeric(df_inf.get("PriceMin", 0), errors="coerce").fillna(0.0)
    price_max = pd.to_numeric(df_inf.get("PriceMax", 0), errors="coerce").fillna(0.0)
    price_mid = (price_min + price_max) / 2.0

    budget = parse_budget_vnd(query_text)
    has_budget = budget > 0
    price_budget_match = np.where(has_budget & (price_mid <= budget), 1.0, 0.0)
    price_distance_norm = np.where(
        has_budget,
        np.maximum(price_mid - budget, 0.0) / np.maximum(budget, 1.0),
        0.0,
    )

    total_view = pd.to_numeric(df_inf.get("TotalView", 0), errors="coerce").fillna(0.0)
    total_fav = pd.to_numeric(df_inf.get("TotalFavourite", 0), errors="coerce").fillna(0.0)
    total_checkin = pd.to_numeric(df_inf.get("TotalCheckins", 0), errors="coerce").fillna(0.0)
    popularity_score = np.log1p(total_view) + 0.5 * np.log1p(total_fav) + 0.5 * np.log1p(total_checkin)

    day_time_features = []
    for _, row in df_inf.iterrows():
        open_map = {day: row.get(f"{day}_open_close", None) for day in FULL_DAYS}
        active_days = row.get("Ngày hoạt động", FULL_DAYS)
        if not isinstance(active_days, list):
            active_days = FULL_DAYS
        day_time_features.append(day_time_match_features(query_text, open_map, active_days))

    day_time_df = pd.DataFrame(day_time_features)

    out = pd.DataFrame({
        "restaurant_id": df_inf["restaurant_id_norm"],
        "restaurant_name": df_inf["restaurant_name"],
        "District": df_inf["District"],
        "Phan_loai_mon": df_inf["Phan_loai_mon"],
        "Price": price_mid.astype(float),
        "Pinecone_score": np.zeros(len(df_inf), dtype=float),
        "BM25_score": np.array(bm25_scores, dtype=float),
        "Semantic_sim_score": semantic_sim_scores.astype(float),
        "District_match_score": df_inf["District"].apply(lambda d: district_match_score(query_text, d)).astype(float),
        "Category_match_score": df_inf["LstCategory_arr"].apply(
            lambda cats: category_match_score(query_text, cats if isinstance(cats, list) else [])
        ).astype(float),
        "Price_budget_match": price_budget_match.astype(float),
        "Price_distance_norm": price_distance_norm.astype(float),
        "Popularity_score": popularity_score.astype(float),
        "Service_match_score": df_inf.apply(
            lambda r: service_match_score(query_text, r.get("Giao tận nơi", ""), r.get("Đặt bàn", "")),
            axis=1,
        ).astype(float),
    })

    out = pd.concat([out, day_time_df], axis=1)

    for col in NUMERICAL_COLS:
        out[col] = pd.to_numeric(out[col], errors="coerce").fillna(0.0)
    for col in CATEGORICAL_COLS:
        out[col] = out[col].fillna("unknown").astype(str).astype("category")

    return out


def rank_top_restaurants(query_text: str, top_k: int = 5) -> pd.DataFrame:
    model = load_ranker_for_inference()
    candidates = build_inference_features(query_text)
    x_pred = candidates[FEATURE_COLS]
    scores = model.predict(x_pred)
    ranked = candidates.copy()
    ranked["score"] = scores
    ranked = ranked.sort_values("score", ascending=False).reset_index(drop=True)
    ranked["rank"] = np.arange(1, len(ranked) + 1)

    return ranked.loc[: top_k - 1, [
        "rank",
        "restaurant_name",
        "score",
        "District",
        "Phan_loai_mon",
        "Price",
    ]]

In [ ]:
# 8) Predict top-5 restaurants for a query
query_text = "quán ăn quận 1 mở cửa tối thứ 6"
top5_result = rank_top_restaurants(query_text, top_k=5)
display(top5_result)